In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros de datos parciales

Se cargan todos los ficheros de servicios que se han ido creando en los notebooks parciales

In [2]:
def load_with_cusec(path):
    df = pd.read_csv(path, encoding="utf-8", dtype={"CUSEC": str})
    # Limpieza y formateo de CUSEC: quitar espacios y asegurar 10 dígitos
    df["CUSEC"] = (
        df["CUSEC"]
        .astype(str)
        .str.strip()
        .str.zfill(10)  # rellena con ceros a la izquierda hasta 10 caracteres
    )
    return df

ceas = load_with_cusec(os.path.join(DATA_OUTPUTS_DA, "accesibilidad_ceas.csv"))
centros_salud = load_with_cusec(os.path.join(DATA_OUTPUTS_DA, "accesibilidad_centros_salud.csv"))
centros_docentes = load_with_cusec(os.path.join(DATA_OUTPUTS_DA, "accesibilidad_centros_docentes.csv"))
servicios_sociales = load_with_cusec(os.path.join(DATA_OUTPUTS_DA, "accesibilidad_servicios_sociales.csv"))

# Cargo el shp para obtener el id:
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)
gdf_secciones["CUSEC"] = (
    gdf_secciones["CUSEC"]
    .astype(str)
    .str.strip()
    .str.zfill(10)  # rellena con ceros a la izquierda hasta 10 caracteres
)


# Integración en un único DF

Se unen todos los parcialicos en un DF unico, por CUSEC

In [3]:
DA = (
    ceas
    .merge(centros_salud, on="CUSEC", how="inner")
    .merge(centros_docentes, on="CUSEC", how="inner")
    .merge(servicios_sociales, on="CUSEC", how="inner")
    .merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        on="CUSEC",
        how="left"
    )
)

DA["Seccion_id"] = DA["Seccion_id"].astype("Int64")

print(DA.info())
DA.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Columns: 104 entries, CUSEC to Seccion_id
dtypes: Int64(1), float64(34), int64(68), object(1)
memory usage: 2.8+ MB
None


,CUSEC,dist_min_ceas_km,n_ceas_1km,n_ceas_5km,n_ceas_15km,n_ceas_30km,disp_ponderada_ceas,dist_min_especializada_km,n_especializada_1km,n_especializada_5km,...,n_voluntariado_15km,n_voluntariado_30km,disp_ponderada_voluntariado,dist_min_personas_mayores_km,n_personas_mayores_1km,n_personas_mayores_5km,n_personas_mayores_15km,n_personas_mayores_30km,disp_ponderada_personas_mayores,Seccion_id
1452,3412007008,0.636161,1,4,4,8,3.2,0.558322,1,1,...,1,1,0.6,1.460271,0,10,10,15,6.5,1453
2383,4201301001,13.174054,0,0,3,5,1.1,11.034250,0,0,...,1,1,0.3,13.174054,0,0,2,3,0.7,2384
2254,4015901001,16.703935,0,0,0,4,0.4,41.159522,0,0,...,0,0,0.0,27.818207,0,0,0,1,0.1,2255
3418,4927101001,10.770893,0,0,1,3,0.5,11.068541,0,0,...,0,0,0.0,42.814626,0,0,0,0,0.0,3419
470,0905907005,0.703077,4,9,9,11,7.2,1.944477,0,3,...,2,2,1.2,4.632347,0,5,5,5,3.0,471


In [4]:
# Crear carpeta si no existe
os.makedirs(DATA_OUTPUTS_DA, exist_ok=True)

# Ruta de salida
ruta_DA = os.path.join(DATA_OUTPUTS_DA, "DA_Seccion.csv")

# Guardar DataFrame con formato controlado
DA.to_csv(
    ruta_DA,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivo guardado en:\n- {ruta_DA}")

✅ Archivo guardado en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DA_Dim_servicios\DA_Seccion.csv
